In [22]:
# Continuous-time prediction tables (apdx:res_utility_of_continuous): same
# setup as 4_prediction.ipynb plus m(t+1) = (1 - eps) s(t) + eps tanh(w . design),
# scored by raw accuracy and flip-and-class balanced accuracy (Appendix C.5).
import numpy as np
import pandas as pd

from utils import (MODELS, S_PREV, FIELD, D_POS, D_NEG, Logistic, adam,
                   sigmoid, load_data, build_xy, two_stage_fit, drives,
                   discrete_rollout, fcba, raw_acc)

runs, banks, P, GCLASS = load_data()
REGIMES = ["subjective", "objective"]
T = 8
print(len(runs), "episodes")

9600 episodes


In [23]:
# continuous fit: MLE of (eps, w), Bernoulli P(s=+1) = (1 + m)/2, full-batch
# Adam (utils.adam), sigmoid-reparameterized eps; small L2 on the field weights
# only, on the raw design (tanh saturation makes their raw scale meaningful).
def fit_continuous(X, s_prev, y01, k, offset=0.0, l2=1e-3, iters=1500):
    """Returns (eps, w) for m = (1 - eps) s_prev + eps tanh(X w + offset);
    the last k columns of X are the couplings (unpenalized, init 0.1)."""
    N, dX = X.shape

    def grad_fn(theta):
        eps, w = sigmoid(theta[0]), theta[1:]
        h = np.tanh(X @ w + offset)
        m = (1.0 - eps) * s_prev + eps * h
        p = 0.5 * (1.0 + np.clip(m, -1.0 + 1e-7, 1.0 - 1e-7))
        w_field = w[1:dX - k]                        # penalized: field sans bias
        nll = (-(y01 * np.log(p) + (1 - y01) * np.log(1 - p)).mean()
               + 0.5 * l2 * float(w_field @ w_field))
        inside = (np.abs(m) < 1.0 - 1e-7).astype(float)
        dL_dm = inside * (p - y01) / (2 * p * (1 - p)) / N
        g = np.zeros_like(theta)
        g[0] = float(np.sum(dL_dm * (h - s_prev))) * eps * (1 - eps)
        g[1:] = X.T @ (dL_dm * eps * (1 - h * h))
        g[2:dX - k + 1] += l2 * w_field
        return nll, g

    theta0 = np.zeros(1 + dX)
    theta0[1 + dX - k:] = 0.1                        # coupling init
    theta = adam(grad_fn, theta0, iters=iters)
    return float(sigmoid(theta[0])), theta[1:]

def continuous_rollout(phi, J_e, s0, eps, w, k, T=8):
    """Free-run carrying the continuous moment m (the mean-field ODE view);
    the reported spin at each step is sign(m)."""
    state = s0.astype(float)
    out = np.empty((len(s0), T, s0.shape[1]), dtype=int)
    for t in range(T):
        X = np.concatenate([phi, drives(J_e, state, k)], axis=-1)
        state = (1.0 - eps) * state + eps * np.tanh(X @ w)
        out[:, t] = np.where(state >= 0.0, 1, -1)
    return out

In [24]:
# the eight table methods; each returns (onestep, rollout) test predictions
def predict_all(x_tr, y_tr, x_te, J_te):
    rows = x_tr.reshape(-1, x_tr.shape[-1])          # pooled train transitions
    parsed = y_tr.reshape(-1) != 0                   # unparsed targets excluded
    y01 = (y_tr.reshape(-1)[parsed] > 0).astype(float)
    E, s0, phi = len(x_te), x_te[:, 0, :, S_PREV], x_te[:, 0, :, FIELD]
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    te_pos, te_neg = x_te[..., D_POS], x_te[..., D_NEG]
    out = {}

    # majority class: the train-majority spin, everywhere
    c = 1 if y01.mean() >= 0.5 else -1
    const = np.full((E, T, 32), c, dtype=int)
    out["Majority Class"] = (const, const)

    # persistence: no change; rollout frozen at s(0)
    out["Persistence"] = (x_te[..., S_PREV].astype(int),
                          np.repeat(s0[:, None].astype(int), T, axis=1))

    # interaction-free: logistic on the static field only (state-independent)
    clf = Logistic().fit(rows[parsed][:, FIELD], y01)
    pred = clf.predict_spin(x_te[..., FIELD])
    out["Interaction-Free"] = (pred, pred)

    # mean-field (Curie-Weiss): [field | population mean s̄(t)]
    def with_sbar(xx):
        sbar = xx[..., S_PREV].mean(axis=-1)
        return np.concatenate([xx[..., FIELD],
                               np.broadcast_to(sbar[..., None, None],
                                               xx.shape[:-1] + (1,))], axis=-1)
    clf = Logistic().fit(with_sbar(x_tr).reshape(-1, 17)[parsed], y01)
    onestep = clf.predict_spin(with_sbar(x_te))
    s, rollout = s0.copy(), np.empty((E, T, 32), dtype=int)
    for t in range(T):
        sbar = np.broadcast_to(s.mean(axis=1)[:, None, None], (E, 32, 1))
        s = clf.predict_spin(np.concatenate([phi, sbar], axis=-1)).astype(float)
        rollout[:, t] = s
    out["Mean-Field"] = (onestep, rollout)

    # discrete update, 1 coupling and (two-stage) 3 couplings
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    onestep = clf.predict_spin(np.concatenate(
        [x_te[..., FIELD], (te_pos + te_neg)[..., None]], axis=-1))
    out["Discrete Update"] = (onestep, discrete_rollout(phi, J_te, s0, clf.w, 1))
    w3 = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                       pos - neg, parsed, y01)
    d3_te = np.concatenate([x_te[..., FIELD], np.stack(
        [te_pos, te_neg, te_pos - te_neg], axis=-1)], axis=-1)
    out["Discrete + Three Couplings"] = (
        np.where(d3_te @ w3 > 0, 1, -1), discrete_rollout(phi, J_te, s0, w3, 3))

    # continuous update, 1 coupling: same design inside tanh, eps carry-over
    sp, y01f = rows[parsed][:, S_PREV], y01
    d1 = np.concatenate([rows[:, FIELD], (pos + neg)[:, None]], axis=-1)
    eps, w = fit_continuous(d1[parsed], sp, y01f, 1)
    m1 = (1 - eps) * x_te[..., S_PREV] + eps * np.tanh(np.concatenate(
        [x_te[..., FIELD], (te_pos + te_neg)[..., None]], axis=-1) @ w)
    out["Continuous Update"] = (np.where(m1 >= 0, 1, -1),
                                continuous_rollout(phi, J_te, s0, eps, w, 1))

    # continuous + 3 couplings, two-stage like the discrete: beta_0 on the
    # unsigned neighborhood first, then the signed betas with beta_0 fixed
    u = pos - neg
    _, w1 = fit_continuous(np.concatenate(
        [rows[:, FIELD], u[:, None]], axis=-1)[parsed], sp, y01f, 1)
    beta_0 = w1[16]
    eps, w2 = fit_continuous(np.concatenate(
        [rows[:, FIELD], np.stack([pos, neg], axis=-1)], axis=-1)[parsed],
        sp, y01f, 2, offset=(beta_0 * u)[parsed])
    w3c = np.concatenate([w2, [beta_0]])
    m1 = (1 - eps) * x_te[..., S_PREV] + eps * np.tanh(d3_te @ w3c)
    out["Continuous + Three Couplings"] = (
        np.where(m1 >= 0, 1, -1), continuous_rollout(phi, J_te, s0, eps, w3c, 3))
    return out

In [25]:
# score both ways: raw accuracy (utils.raw_acc) and the paper's flip-and-class
# balanced accuracy (utils.fcba, Appendix C.5). Majority Class scores exactly
# 50 under the balanced metric, so the balanced tables drop it.
METHOD_ROWS = ["Majority Class", "Persistence", "Interaction-Free",
               "Mean-Field", "Discrete Update", "Discrete + Three Couplings",
               "Continuous Update", "Continuous + Three Couplings"]
METRIC_ROWS = {"raw": METHOD_ROWS,
               "balanced": [m for m in METHOD_ROWS if m != "Majority Class"]}
METRIC_NAMES = {"raw": "raw accuracy",
                "balanced": "flip-and-class balanced accuracy"}
SPLITS = [("In-d.", "seen"), ("Out-d.", "fresh")]
results = {}
for regime in REGIMES:
    for model, label, short in MODELS:
        x, y, ep = build_xy(model, regime, runs, banks, P, GCLASS)
        train = (ep["split"] == "train") & np.isin(ep["gcls"], ["seen", "train_only"])
        test = (ep["split"] == "test") & np.isin(ep["gcls"], ["seen", "fresh"])
        preds = predict_all(x[train], y[train], x[test], ep["J"][test])
        y_te, s0, gc = y[test], x[test][:, 0, :, S_PREV], ep["gcls"][test]
        score = {"raw": lambda p, m: raw_acc(p, y_te, m),
                 "balanced": lambda p, m: fcba(p, y_te, s0, m)}
        for method, (onestep, rollout) in preds.items():
            for metric, fn in score.items():
                results[metric, regime, label, method] = {
                    split: (fn(onestep, gc == g), fn(rollout, gc == g))
                    for split, g in SPLITS}
        print(f"{label:13s} {regime:10s}  done")

GPT-4o-mini   subjective  done
Gemma-3n-E4B  subjective  done
Qwen3.5-9B    subjective  done
Llama-3-8B    subjective  done
GPT-4o-mini   objective   done
Gemma-3n-E4B  objective   done
Qwen3.5-9B    objective   done
Llama-3-8B    objective   done


In [26]:
# the tables: one-step (rollout) accuracy per cell, per metric
def cell(metric, regime, label, method, split):
    o, r = results[metric, regime, label, method][split]
    return f"{o:.1f} ({r:.1f})"

for metric in ("raw", "balanced"):
    for regime in REGIMES:
        print(f"=== {regime.capitalize()} questions — {METRIC_NAMES[metric]} ===")
        display(pd.DataFrame({m: {(l, sp): cell(metric, regime, l, m, sp)
                                  for _, l, _ in MODELS for sp, _ in SPLITS}
                              for m in METRIC_ROWS[metric]}).T)

=== Subjective questions — raw accuracy ===


GPT-4o-mini              Gemma-3n-E4B  \
                                    In-d.       Out-d.        In-d.   
Majority Class                64.0 (64.0)  61.9 (61.9)  54.0 (54.0)   
Persistence                   76.9 (73.2)  73.7 (71.6)  81.3 (70.9)   
Interaction-Free              53.7 (53.7)  50.7 (50.7)  64.3 (64.3)   
Mean-Field                    67.7 (60.9)  66.7 (62.1)  78.3 (73.5)   
Discrete Update               76.6 (69.2)  74.5 (67.3)  64.8 (64.1)   
Discrete + Three Couplings    86.3 (77.4)  85.4 (76.7)  81.1 (74.2)   
Continuous Update             82.7 (73.0)  80.4 (70.3)  81.2 (67.4)   
Continuous + Three Couplings  89.2 (80.0)  87.6 (80.3)  84.5 (76.7)   

                                            Qwen3.5-9B               \
                                   Out-d.        In-d.       Out-d.   
Majority Class                55.9 (55.9)  50.5 (50.5)  49.2 (49.2)   
Persistence                   81.1 (70.6)  75.3 (75.2)  74.3 (74.8)   
Interaction-Free              62.6 (62.6)  48.4 (48.4)  46.3 (46.3)   
Mean-Field                    77.5 (73.9)  73.6 (71.7)  74.0 (73.1)   
Discrete Update               62.4 (62.0)  63.5 (56.9)  58.3 (54.1)   
Discrete + Three Couplings    79.9 (70.4)  83.3 (75.1)  80.3 (71.8)   
Continuous Update             80.3 (65.5)  74.5 (58.3)  70.7 (55.0)   
Continuous + Three Couplings  83.6 (72.4)  85.4 (75.6)  82.4 (73.4)   

                               Llama-3-8B               
                                    In-d.       Out-d.  
Majority Class                87.2 (87.2)  87.4 (87.4)  
Persistence                   85.1 (75.3)  85.6 (74.4)  
Interaction-Free              49.6 (49.6)  48.9 (48.9)  
Mean-Field                    86.4 (74.9)  86.2 (73.9)  
Discrete Update               55.8 (54.2)  54.0 (53.1)  
Discrete + Three Couplings    90.8 (81.2)  90.4 (81.2)  
Continuous Update             75.0 (61.2)  73.8 (60.2)  
Continuous + Three Couplings  94.2 (88.2)  93.7 (88.0)

=== Objective questions — raw accuracy ===


GPT-4o-mini              Gemma-3n-E4B  \
                                    In-d.       Out-d.        In-d.   
Majority Class                47.2 (47.2)  46.5 (46.5)  58.6 (58.6)   
Persistence                   63.2 (58.6)  62.4 (57.7)  86.6 (78.6)   
Interaction-Free              48.9 (48.9)  47.9 (47.9)  61.6 (61.6)   
Mean-Field                    69.5 (56.2)  69.3 (56.6)  90.2 (82.9)   
Discrete Update               68.8 (60.7)  66.2 (59.2)  61.2 (61.2)   
Discrete + Three Couplings    86.7 (67.4)  85.5 (64.9)  91.7 (81.8)   
Continuous Update             70.9 (57.9)  68.3 (56.2)  86.6 (69.1)   
Continuous + Three Couplings  87.2 (70.0)  85.6 (67.9)  92.4 (82.0)   

                                            Qwen3.5-9B               \
                                   Out-d.        In-d.       Out-d.   
Majority Class                58.5 (58.5)  49.1 (49.1)  49.4 (49.4)   
Persistence                   87.1 (79.3)  81.8 (63.9)  82.4 (63.5)   
Interaction-Free              62.0 (62.0)  43.5 (43.5)  44.7 (44.7)   
Mean-Field                    90.4 (84.2)  87.3 (69.6)  87.6 (68.6)   
Discrete Update               61.6 (61.9)  47.9 (46.3)  47.6 (47.1)   
Discrete + Three Couplings    91.5 (81.6)  91.3 (66.8)  90.9 (65.4)   
Continuous Update             87.1 (69.8)  73.8 (47.2)  73.3 (48.2)   
Continuous + Three Couplings  92.6 (81.9)  91.5 (67.1)  91.1 (65.4)   

                               Llama-3-8B               
                                    In-d.       Out-d.  
Majority Class                79.4 (79.4)  77.9 (77.9)  
Persistence                   75.7 (66.2)  75.8 (65.6)  
Interaction-Free              79.4 (79.4)  77.9 (77.9)  
Mean-Field                    83.2 (75.0)  82.2 (72.4)  
Discrete Update               79.3 (79.0)  77.6 (77.3)  
Discrete + Three Couplings    89.3 (78.9)  88.4 (77.5)  
Continuous Update             79.7 (79.4)  78.1 (77.7)  
Continuous + Three Couplings  89.4 (79.9)  88.4 (78.3)

=== Subjective questions — flip-and-class balanced accuracy ===


GPT-4o-mini              Gemma-3n-E4B  \
                                    In-d.       Out-d.        In-d.   
Persistence                   50.0 (64.5)  50.0 (64.0)  50.0 (59.3)   
Interaction-Free              50.6 (50.6)  48.6 (48.6)  60.8 (60.8)   
Mean-Field                    57.9 (56.6)  59.3 (57.9)  68.3 (64.6)   
Discrete Update               77.7 (69.4)  75.7 (67.1)  62.4 (61.0)   
Discrete + Three Couplings    86.2 (76.4)  86.2 (77.1)  75.5 (68.0)   
Continuous Update             79.3 (66.0)  77.5 (65.0)  61.8 (60.4)   
Continuous + Three Couplings  87.8 (76.8)  87.3 (79.0)  72.7 (67.2)   

                                            Qwen3.5-9B               \
                                   Out-d.        In-d.       Out-d.   
Persistence                   50.0 (59.3)  50.0 (65.1)  50.0 (65.1)   
Interaction-Free              59.6 (59.6)  49.1 (49.1)  47.5 (47.5)   
Mean-Field                    67.9 (64.9)  65.2 (64.4)  65.7 (65.5)   
Discrete Update               61.3 (59.9)  64.6 (57.4)  60.5 (54.8)   
Discrete + Three Couplings    75.3 (66.8)  81.3 (70.8)  78.8 (69.2)   
Continuous Update             61.3 (59.2)  66.4 (55.6)  63.8 (53.2)   
Continuous + Three Couplings  72.7 (65.5)  81.3 (70.3)  78.7 (68.8)   

                               Llama-3-8B               
                                    In-d.       Out-d.  
Persistence                   50.0 (63.2)  50.0 (61.7)  
Interaction-Free              47.7 (47.7)  45.6 (45.6)  
Mean-Field                    54.8 (56.8)  53.5 (57.4)  
Discrete Update               54.0 (49.7)  51.7 (47.8)  
Discrete + Three Couplings    82.6 (68.4)  80.8 (68.3)  
Continuous Update             58.7 (51.4)  58.6 (50.0)  
Continuous + Three Couplings  83.4 (68.5)  82.5 (70.6)

=== Objective questions — flip-and-class balanced accuracy ===


GPT-4o-mini              Gemma-3n-E4B  \
                                    In-d.       Out-d.        In-d.   
Persistence                   50.0 (55.6)  50.0 (55.0)  50.0 (61.8)   
Interaction-Free              49.8 (49.8)  49.3 (49.3)  59.0 (59.0)   
Mean-Field                    65.4 (55.1)  65.3 (55.5)  72.7 (67.3)   
Discrete Update               70.1 (61.5)  67.8 (60.3)  59.7 (58.9)   
Discrete + Three Couplings    86.3 (68.2)  85.0 (65.9)  80.5 (68.5)   
Continuous Update             69.8 (57.4)  67.6 (56.1)  50.0 (56.8)   
Continuous + Three Couplings  86.5 (70.6)  85.0 (68.6)  79.0 (67.5)   

                                            Qwen3.5-9B               \
                                   Out-d.        In-d.       Out-d.   
Persistence                   50.0 (62.2)  50.0 (53.9)  50.0 (53.2)   
Interaction-Free              59.1 (59.1)  44.6 (44.6)  45.4 (45.4)   
Mean-Field                    72.7 (68.3)  73.6 (62.1)  73.4 (61.1)   
Discrete Update               59.7 (59.1)  50.6 (47.8)  49.8 (48.3)   
Discrete + Three Couplings    80.9 (68.5)  86.3 (65.7)  85.2 (64.2)   
Continuous Update             50.0 (57.2)  53.5 (44.8)  52.2 (45.2)   
Continuous + Three Couplings  79.3 (67.5)  86.2 (65.2)  85.1 (63.7)   

                               Llama-3-8B               
                                    In-d.       Out-d.  
Persistence                   50.0 (57.0)  50.0 (56.8)  
Interaction-Free              50.0 (50.0)  50.0 (50.0)  
Mean-Field                    64.7 (61.2)  64.5 (60.5)  
Discrete Update               50.9 (50.0)  50.6 (50.0)  
Discrete + Three Couplings    77.4 (62.9)  76.4 (60.8)  
Continuous Update             51.2 (50.1)  51.0 (50.1)  
Continuous + Three Couplings  77.7 (60.3)  76.5 (59.1)

In [27]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

# This cell has two modes: regenerate from in-memory results when available,
# or render the saved TeX export after a kernel restart.
def table_path(filename):
    local = Path(filename)
    repository = Path("res") / filename
    if local.exists():
        return local
    if repository.exists():
        return repository
    return repository if Path("res").is_dir() else local

latex_path = table_path("prediction_continuous_tables.tex")
has_results = all(name in globals() for name in (
    "results", "METRIC_ROWS", "METRIC_NAMES", "MODELS", "REGIMES", "SPLITS"))

if has_results:
    latex_row = {
        "Discrete + Three Couplings": r"\quad + Three Couplings",
        "Continuous + Three Couplings": r"\quad + Three Couplings",
    }
    latex_tables = []
    for metric in ("raw", "balanced"):
        rows_m = METRIC_ROWS[metric]
        for regime in REGIMES:
            lines = [r"\begin{table}[h!]", r"\centering",
                     r"\setlength{\tabcolsep}{3pt}",
                     r"\resizebox{\textwidth}{!}{%",
                     r"\begin{tabular}{l cc cc cc cc}", r"\toprule",
                     " & " + " & ".join(
                         rf"\multicolumn{{2}}{{c}}{{{label}}}"
                         for _, label, _ in MODELS) + r" \\",
                     "".join(rf"\cmidrule(lr){{{2 * i}-{2 * i + 1}}}"
                             for i in range(1, 5)),
                     "Method & " + " & ".join(
                         ["In-d. & Out-d."] * 4) + r" \\",
                     r"\midrule"]
            for method in rows_m:
                cells = []
                for _, label, _ in MODELS:
                    for split, _ in SPLITS:
                        one_step, rollout = results[
                            metric, regime, label, method][split]
                        best_one = max(results[
                            metric, regime, label, candidate][split][0]
                                       for candidate in rows_m)
                        best_rollout = max(results[
                            metric, regime, label, candidate][split][1]
                                           for candidate in rows_m)
                        one_text = (rf"\textbf{{{one_step:.1f}}}"
                                    if one_step == best_one
                                    else f"{one_step:.1f}")
                        rollout_text = (rf"\textbf{{{rollout:.1f}}}"
                                        if rollout == best_rollout
                                        else f"{rollout:.1f}")
                        cells.append(f"{one_text} ({rollout_text})")
                lines.append(
                    latex_row.get(method, method) + " & "
                    + " & ".join(cells) + r" \\")
                if method in ("Mean-Field", "Discrete + Three Couplings"):
                    lines.append(r"\midrule")
            lines += [r"\bottomrule", r"\end{tabular}%", "}",
                      rf"\caption{{\textit{{Predictions on "
                      rf"{regime.capitalize()} Questions.}} "
                      rf"{METRIC_NAMES[metric].capitalize()}.}}",
                      r"\end{table}"]
            latex_tables.append("\n".join(lines))
    latex_fragment = "\n\n".join(latex_tables)
    standalone_latex = "\n".join([
        r"\documentclass{article}",
        r"\usepackage{booktabs}",
        r"\usepackage{graphicx}",
        r"\begin{document}",
        latex_fragment,
        r"\end{document}",
    ])
    latex_path.write_text(standalone_latex + "\n", encoding="utf-8")
    status = f"Regenerated {latex_path} from in-memory results."
else:
    if not latex_path.exists():
        raise FileNotFoundError(
            f"Saved table file not found: {latex_path}.")
    standalone_latex = latex_path.read_text(encoding="utf-8")
    latex_tables = re.findall(
        r"\\begin\{table\}.*?\\end\{table\}",
        standalone_latex, flags=re.DOTALL)
    status = f"Rendered saved results from {latex_path}; no models were run."

if len(latex_tables) != 4:
    raise ValueError(f"Expected four tables in {latex_path}, found {len(latex_tables)}.")

preview_columns = pd.MultiIndex.from_product([
    ("GPT-4o-mini", "Gemma-3n-E4B", "Qwen3.5-9B", "Llama-3-8B"),
    ("In-d.", "Out-d."),
])

def clean_latex(text):
    text = re.sub(r"\\textbf\{([^{}]+)\}", r"\1", text)
    return (text.replace(r"\quad", "")
            .removesuffix(r"\\").strip())

def preview_rows(table_text):
    rows = []
    previous = ""
    for line in table_text.splitlines():
        if "&" not in line or not line.rstrip().endswith(r"\\"):
            continue
        cells = [clean_latex(cell) for cell in line.split("&")]
        if len(cells) != 9 or cells[0] == "Method":
            continue
        method = cells[0]
        if method == "+ Three Couplings":
            method = previous.split()[0] + " + Three Couplings"
        rows.append([method] + cells[1:])
        previous = method
    return rows

table_titles = (
    "Subjective questions — Raw accuracy",
    "Objective questions — Raw accuracy",
    "Subjective questions — Flip-and-class balanced accuracy",
    "Objective questions — Flip-and-class balanced accuracy",
)
for title, table_text in zip(table_titles, latex_tables):
    rows = preview_rows(table_text)
    frame = pd.DataFrame(
        [row[1:] for row in rows],
        index=[row[0] for row in rows],
        columns=preview_columns)
    display(frame.style.set_caption(title))

print(status)


Regenerated prediction_continuous_tables.tex from in-memory results.
